# 10.7 圖片展開後，答案目標如何跟著移動？


一串訊息原本只有一個「圖片」格，實際圖片卻要放三條特徵。替換後，後面的助手角色與回答一起往後移兩格。除了移輸入，也要移對應目標，讓助手位置仍預測答案第一個字。

為了核對位置，本例手設答案為字母 A，再接結束標記。用 1 表開頭、5 表圖片、4 表助手、73 表 A、2 表結束；A 的 ASCII byte 是 65，加 8 得 ID 73。圖片特徵不對應要生成的字，目標填 `-100` 表示不計文字代價，但它們仍供後面讀取。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from tiny_perceptron.multimodal import expand_modalities

embedding = nn.Embedding(264, 8)
ids = torch.tensor([1, 5, 4, 73, 2])
labels = torch.tensor([-100, -100, -100, 73, 2])
features = {5: torch.randn(3, 8)}
x, y = expand_modalities(ids, labels, embedding, features, {5})
print("輸入特徵", tuple(x.shape))
print("下一格目標", y.tolist())

原五格展開為「開頭、圖1、圖2、圖3、助手、73、結束」七格，完整目標是 `[-100,-100,-100,-100,-100,73,2]`。然後只做一次下一位置對齊：輸入去掉末格，目標去掉首格，得到六格輸入 `(1,6,8)` 與 `[[-100,-100,-100,-100,73,2]]`。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-10-image-targets.svg")))

圖裡將前四個上下文位置合成一列顯示，仍是開頭加三條圖特徵。索引 4 的輸入是助手、目標是 73；索引 5 的輸入是 73、目標是結束 2。不是先把答案給助手抄，也不是每個圖片位置都要輸出一個字。

後續位置編號和防偷看遮罩都按這六格建立：位置為 0–5，遮罩比較六個讀取位置與六個被讀位置。不能沿用原五格，或把未錯位七格當成實際輸入長度。先展開、再對齊，讓這個責任只發生一次。

練習將三條圖片特徵改成五條，預期輸入長度 8、前六個目標忽略，末兩個仍是 73 與 2。回答內容沒變，改變的是它在序列中的位置。

<details>
<summary>回顧與查證</summary>

可回顧：[10.6接頭輸出](https://birdhackor.github.io/tiny-perceptron-vlm/10.6.html)、[7.4下一位置的答案對齊](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)。

</details>
